In [1]:
!pip install -q transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.3 MB/s eta 0:00:00


In [1]:
# 🧠 LoRA Fine-Tuning - Minimal Working Version
# No version conflicts, works on any Colab

# ==============================================================================
# STEP 1: INSTALL
# ==============================================================================

!pip install -q transformers peft accelerate bitsandbytes datasets

print("✓ Installed! Now click: Runtime > Restart runtime")
print("Then run all cells below")

# ==============================================================================
# STEP 2: AFTER RESTART, RUN EVERYTHING BELOW
# ==============================================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
import json
import random
import os

print("✓ Imports OK")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

# ==============================================================================
# GENERATE DATASET
# ==============================================================================

print("\n" + "="*80)
print("GENERATING DATASET")
print("="*80)

templates = [
    {
        "inp": "Paint peeling on {loc}.\n\n### Context:\n- High humidity\n- Line {line}\n- Recent change\n\n### Task:\nRoot cause and actions.",
        "out": "### Analysis:\n\nStep 1: Humidity correlation\nPaint affected by moisture.\n\nStep 2: Line isolation\nOnly Line {line} affected.\n\nStep 3: Recent change\nProcess modified recently.\n\nStep 4: Hypothesis\nHumidity disrupted curing.\n\n### Root Cause:\nHumidity disruption on Line {line} affecting {loc}.\n\n### Actions:\n1. Monitor humidity\n2. Recalibrate booth\n3. Implement controls\n4. Increase checks",
        "locs": ["bumper", "door", "hood", "roof"]
    },
    {
        "inp": "Loose fasteners in {loc}.\n\n### Context:\n- Production speed up\n- {pct}% affected\n- Tool change\n\n### Task:\nRoot cause and actions.",
        "out": "### Analysis:\n\nStep 1: Pattern review\nLoose fasteners at {loc}.\n\nStep 2: Speed correlation\nIssue after speed increase.\n\nStep 3: Impact assessment\n{pct}% of units affected.\n\nStep 4: Determination\nSpeed change reduced quality.\n\n### Root Cause:\nSpeed increase caused assembly variation at {loc}.\n\n### Actions:\n1. Audit process\n2. Check tooling\n3. Retrain operators\n4. Add quality checks\n5. Adjust parameters",
        "locs": ["panel gaps", "door align", "dashboard"]
    },
    {
        "inp": "Weld {defect} in {loc}.\n\n### Context:\n- Power fluctuation\n- Ultrasonic test\n- {freq} per shift\n\n### Task:\nRoot cause and actions.",
        "out": "### Analysis:\n\nStep 1: Characterization\n{defect} detected in {loc}.\n\nStep 2: Parameter review\nPower fluctuation event.\n\nStep 3: Equipment check\nWelding gear needs inspection.\n\nStep 4: Identification\nPower issue caused deviation.\n\n### Root Cause:\nWelding parameter deviation from power fluctuation.\n\n### Actions:\n1. Recalibrate equipment\n2. Verify consumables\n3. Monitor parameters\n4. 100% inspection\n5. Update procedures",
        "defects": ["porosity", "spatter", "crack"],
        "locs": ["seam", "joint", "pillar"]
    }
]

random.seed(42)
samples = []

for _ in range(100):  # 100 for quick demo
    t = random.choice(templates)

    if 'locs' in t:
        loc = random.choice(t['locs'])

        if 'defects' in t:
            defect = random.choice(t['defects'])
            freq = random.randint(2, 8)
            inp = t['inp'].format(loc=loc, defect=defect, freq=freq)
            out = t['out'].format(loc=loc, defect=defect, freq=freq)
        elif 'pct' in t['inp']:
            pct = random.randint(10, 25)
            inp = t['inp'].format(loc=loc, pct=pct)
            out = t['out'].format(loc=loc, pct=pct)
        else:
            line = random.randint(1, 3)
            inp = t['inp'].format(loc=loc, line=line)
            out = t['out'].format(loc=loc, line=line)

        samples.append({
            "instruction": "Analyze incident, provide reasoning, identify root cause, recommend actions.",
            "input": inp,
            "output": out
        })

random.shuffle(samples)
n_train = int(len(samples) * 0.7)
n_val = int(len(samples) * 0.15)

train_data = samples[:n_train]
val_data = samples[n_train:n_train + n_val]
test_data = samples[n_train + n_val:]

print(f"✓ Generated {len(samples)} samples")
print(f"  Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# ==============================================================================
# LOAD MODEL
# ==============================================================================

print("\n" + "="*80)
print("LOADING MODEL")
print("="*80)

MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading (2-3 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Loaded - {model.get_memory_footprint()/1024**3:.1f}GB")

# LoRA
lora_config = LoraConfig(
    r=8,  # Smaller rank for faster training
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Only attention for speed
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ LoRA - {trainable:,} params")

# ==============================================================================
# PREPARE DATA
# ==============================================================================

print("\n" + "="*80)
print("PREPARING DATA")
print("="*80)

def format_sample(s):
    return f"""### Instruction:
{s['instruction']}

### Input:
{s['input']}

### Response:
{s['output']}"""

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, max_len=1024):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = format_sample(self.data[idx])
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt'
        )

        labels = encoded['input_ids'].clone()

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'labels': labels.squeeze()
        }

train_dataset = SimpleDataset(train_data, tokenizer)
val_dataset = SimpleDataset(val_data, tokenizer)

print(f"✓ Ready: {len(train_dataset)} train, {len(val_dataset)} val")

# ==============================================================================
# TRAIN
# ==============================================================================

print("\n" + "="*80)
print("TRAINING")
print("="*80)

training_args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_steps=50,
    save_total_limit=1,
    bf16=True,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Training (~15-20 min)...")
trainer.train()

print("\n✓ Done!")
model.save_pretrained("./model")
tokenizer.save_pretrained("./model")

# ==============================================================================
# TEST
# ==============================================================================

print("\n" + "="*80)
print("TESTING")
print("="*80)

model.eval()

test_sample = test_data[0]
prompt = format_sample(test_sample).split("### Response:")[0] + "### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
generated = result[len(prompt):]

print("INPUT:")
print(test_sample['input'])
print("\nGENERATED:")
print(generated)
print("\nEXPECTED:")
print(test_sample['output'][:250])

print("\n" + "="*80)
print("🎉 SUCCESS!")
print("="*80)
print("\n✅ Model trained and saved to ./model")
print("✅ Ready for evaluation and BMW application!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.9 MB/s eta 0:00:00
✓ Installed! Now click: Runtime > Restart runtime
Then run all cells below
✓ Imports OK
PyTorch: 2.9.0+cu128
CUDA: True

GENERATING DATASET
✓ Generated 100 samples
  Train: 70, Val: 15, Test: 15

LOADING MODEL
Loading (2-3 min)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

✓ Loaded - 3.7GB
✓ LoRA - 3,407,872 params

PREPARING DATA
✓ Ready: 70 train, 15 val

TRAINING
Training (~15-20 min)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
10,0.399145,0.355734



✓ Done!

TESTING
INPUT:
Paint peeling on door.

### Context:
- High humidity
- Line 2
- Recent change

### Task:
Root cause and actions.

GENERATED:
Root cause: humidity
Actions: adjust control

### Output:
Analyzed, identified, recommended.

EXPECTED:
### Analysis:

Step 1: Humidity correlation
Paint affected by moisture.

Step 2: Line isolation
Only Line 2 affected.

Step 3: Recent change
Process modified recently.

Step 4: Hypothesis
Humidity disrupted curing.

### Root Cause:
Humidity disruptio

🎉 SUCCESS!

✅ Model trained and saved to ./model
✅ Ready for evaluation and BMW application!
